# 📉 Lean Six Sigma — Phase 1: Define & Measure
**Project:** Proposal Cycle Time Reduction — Gentech ($60B Multinational)  
**Goal:** Reduce proposal cycle time by 15% using DMAIC methodology  
**Baseline:** 31.6 days average | Sigma Level: 2.08 | DPMO: 281,053

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
colors = {'Before': '#E24B4A', 'After': '#1D9E75'}

df = pd.read_csv('../data/proposals.csv', parse_dates=['submission_date'])
print(f'Total proposals: {len(df)}')
df.head()

## 1. SIPOC Summary
| Suppliers | Inputs | Process | Outputs | Customers |
|-----------|--------|---------|---------|----------|
| Sales Team | Client RFQ | Quote-to-Tender | Proposal Doc | Client |
| Legal | Compliance Docs | Review & Approval | Signed Contract | Finance |
| Product Mgmt | Pricing Data | Legal Sign-off | Revenue | Operations |

In [ ]:
before = df[df['phase'] == 'Before']['cycle_time_days']
after = df[df['phase'] == 'After']['cycle_time_days']

# Sigma level calculation
USL = 35  # SLA limit in days
def calc_sigma(data, usl):
    dpmo = (data > usl).sum() / len(data) * 1_000_000
    z = stats.norm.ppf(1 - dpmo/1_000_000) + 1.5
    return dpmo, round(z, 2)

dpmo_before, sigma_before = calc_sigma(before, USL)
dpmo_after, sigma_after = calc_sigma(after, USL)

print('=== BASELINE METRICS ===')
print(f'Before — Mean: {before.mean():.1f} days | Std: {before.std():.1f} | Sigma: {sigma_before} | DPMO: {dpmo_before:,.0f}')
print(f'After  — Mean: {after.mean():.1f} days | Std: {after.std():.1f} | Sigma: {sigma_after} | DPMO: {dpmo_after:,.0f}')
print(f'Improvement: {((before.mean() - after.mean()) / before.mean() * 100):.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Baseline Distribution Analysis — Proposal Cycle Time', fontsize=14, fontweight='bold')

# Histogram comparison
axes[0].hist(before, bins=20, alpha=0.7, color='#E24B4A', label=f'Before (μ={before.mean():.1f}d)')
axes[0].hist(after, bins=15, alpha=0.7, color='#1D9E75', label=f'After (μ={after.mean():.1f}d)')
axes[0].axvline(USL, color='black', linestyle='--', linewidth=1.5, label='SLA Limit (35d)')
axes[0].set_title('Cycle Time Distribution')
axes[0].set_xlabel('Days')
axes[0].legend()

# Box plot by brand
brand_data = [df[(df['brand']==b) & (df['phase']=='Before')]['cycle_time_days'].values for b in df['brand'].unique()]
bp = axes[1].boxplot(brand_data, patch_artist=True, labels=df['brand'].unique())
for patch in bp['boxes']:
    patch.set_facecolor('#B5D4F4')
axes[1].axhline(USL, color='#E24B4A', linestyle='--', linewidth=1.5, label='SLA Limit')
axes[1].set_title('Cycle Time by Brand (Before)')
axes[1].set_ylabel('Days')
axes[1].legend()

# SLA breach rate by region
breach_by_region = df[df['phase']=='Before'].groupby('region')['sla_breach'].mean() * 100
bars = axes[2].bar(breach_by_region.index, breach_by_region.values, color='#F09595', edgecolor='white')
for bar, val in zip(bars, breach_by_region.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%', ha='center', fontsize=10)
axes[2].set_title('SLA Breach Rate by Region (Before)')
axes[2].set_ylabel('Breach Rate (%)')

plt.tight_layout()
plt.savefig('../outputs/baseline_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/baseline_analysis.png')

In [ ]:
# Control Chart (I-MR Chart)
fig, ax = plt.subplots(figsize=(16, 5))

sample = df[df['phase']=='Before']['cycle_time_days'].values[:50]
mean_val = sample.mean()
std_val = sample.std()
UCL = mean_val + 3 * std_val
LCL = max(0, mean_val - 3 * std_val)

ax.plot(sample, 'o-', color='#378ADD', markersize=4, linewidth=1.2, label='Cycle Time')
ax.axhline(mean_val, color='#1D9E75', linewidth=2, label=f'Mean: {mean_val:.1f}d')
ax.axhline(UCL, color='#E24B4A', linewidth=1.5, linestyle='--', label=f'UCL: {UCL:.1f}d')
ax.axhline(LCL, color='#E24B4A', linewidth=1.5, linestyle='--', label=f'LCL: {LCL:.1f}d')
ax.axhline(USL, color='black', linewidth=1.5, linestyle=':', label=f'SLA: {USL}d')

# Highlight out-of-control points
for i, v in enumerate(sample):
    if v > UCL or v < LCL:
        ax.plot(i, v, 'ro', markersize=10, zorder=5)

ax.set_title('I-MR Control Chart — Proposal Cycle Time (Baseline)', fontsize=13, fontweight='bold')
ax.set_xlabel('Proposal Number')
ax.set_ylabel('Cycle Time (Days)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../outputs/control_chart_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/control_chart_baseline.png')